# NB4 · Açıklanabilirlik ve açıklamanın eleştirisi
### Explainability, and the critique of the explanation

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr  
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

---

Bu defter iki yöntem kullanmaktadır ve aralarındaki fark dersin kendisidir.

**Permütasyon önemi** model bağımsızdır. Bir öznitelik karıştırıldığında başarımın ne
kadar düştüğünü ölçer, yani modelin o özniteliğe ne kadar dayandığını gösterir.

**Doğrusal katkı ayrıştırması** yaklaşık değil kesindir. Lojistik regresyonda log odds,
katsayı çarpı öznitelik değerlerinin toplamıdır; dolayısıyla tek bir tahmindeki her
özniteliğin katkısı doğrudan okunabilir. Tahmin edilen bir şey yoktur.

Kesinlik burada bilinçli bir seçimdir. Doğrusal olmayan bir modele geçildiğinde bu
ayrıştırma artık mümkün olmaz ve SHAP gibi sonradan yapılan bir yaklaşıklama gerekir.
Katılımcının önce kesin sürümü görmesi, yaklaşıklamanın neyi yaklaşıkladığını anlaması
içindir.


## 0. Kurulum · Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['mimic_web.py', 'evaluate.py', 'explain.py', 'safety.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import evaluate as ev
import explain as ex
import pipeline as pl

state = pl.prepare()
model, test, features = state['model'], state['test'], state['features']
y_test, probabilities = state['y_test'], state['probabilities']


## 1. Küresel açıklama · Global explanation

Aşağıdaki tabloda `sd` sütunu en az `importance` sütunu kadar önemlidir. Küçük bir
kohortta tekrarlar arasındaki yayılım, komşu öznitelikler arasındaki farktan büyük
olmaktadır. Bu durumda sıralamanın kendisi kararsızdır ve bir sıralama olarak rapor
edilmemelidir. `ranking_is_stable` sütunu bunu açıkça işaretlemektedir.


In [ ]:
importance = ex.permutation_global(model, test[features], y_test, n_repeats=20)
importance.round(4)


In [ ]:
fig = ex.plot_global(importance, title='Permütasyon önemi · ROC AUC düşüşü')


Lojistik regresyonda katsayılar da doğrudan okunabilmektedir. Katsayılar ölçeklenmiş
öznitelik uzayında olduğu için birbirleriyle karşılaştırılabilir; `odds_ratio` sütunu
bir standart sapmalık artışın olasılık oranına etkisini vermektedir.


In [ ]:
ex.linear_coefficients(model, top=12).round(3)


## 2. Yerel açıklama: Üç vaka · Local explanation, three cases

Bir doğru pozitif, bir yanlış pozitif ve bir yanlış negatif seçilmektedir. Önemli olan
**yanlış pozitiftir.** Yanlış bir tahmini makul gösteren bir açıklama, açıklanabilirliğin
hatayı aklama mekanizmasıdır ve bunu bir kez görmek kavramın tanımını öğrenmekten
daha değerlidir.


In [ ]:
threshold = ev.threshold_for_sensitivity(y_test, probabilities, target=0.80)
cases = ex.pick_cases(y_test, probabilities, threshold)

print(f'Eşik: {threshold:.3f}')
for name, index in cases.items():
    if index is None:
        print(f'{name:<16} bu test kümesinde bulunamadı')
    else:
        print(f'{name:<16} satır {index}, olasılık {probabilities[index]:.3f}')


In [ ]:
case = cases['true_positive']
if case is not None:
    explanation = ex.explain_case(model, test[features], case)
    display(explanation['contributions'].round(3))
    ex.plot_case(explanation, title=f'DOĞRU POZİTİF · olasılık {explanation["probability"]:.3f}')


In [ ]:
case = cases['false_positive']
if case is not None:
    explanation = ex.explain_case(model, test[features], case)
    display(explanation['contributions'].round(3))
    ex.plot_case(explanation, title=f'YANLIŞ POZİTİF · olasılık {explanation["probability"]:.3f}')


In [ ]:
case = cases['false_negative']
if case is not None:
    explanation = ex.explain_case(model, test[features], case)
    display(explanation['contributions'].round(3))
    ex.plot_case(explanation, title=f'YANLIŞ NEGATİF · olasılık {explanation["probability"]:.3f}')


## 3. Kesinlik denetimi · The exactness check

Aşağıdaki hücre, katkıların gerçekten bir özdeşlik olduğunu göstermektedir: Katkıların
toplamı artı kesişim, log odds değerini vermekte; onun sigmoidi ise modelin ürettiği
olasılığa eşit olmaktadır. Bir yaklaşıklamada bu eşitlik tutmaz.


In [ ]:
case = cases['false_positive'] or cases['true_positive']
explanation = ex.explain_case(model, test[features], case, top=10**6)

log_odds = explanation['log_odds']
from_contributions = 1 / (1 + np.exp(-log_odds))
from_model = explanation['probability']

print(f'Katkılardan hesaplanan olasılık : {from_contributions:.10f}')
print(f'Modelin ürettiği olasılık       : {from_model:.10f}')
print(f'Fark                            : {abs(from_contributions - from_model):.2e}')


## 4. Açıklama eleştirisi · Explanation critique

Yukarıdaki üç grafiğe bakarak aşağıdaki soruları kendiniz cevaplayınız, ardından aynı
soruları asistana beşinci istemin ikinci bölümü olarak yöneltiniz.

**Sızıntı belirtisi var mı?** Yüksek katkılı özniteliklerden biri, klinik bir sinyal
yerine verinin kaydediliş biçiminden kaynaklanan bir artefakta benziyor mu? Bu kohortta
ölçüm sayısı sütunları (`_n` ile bitenler) buna adaydır: Bir hastadan ilk altı saatte
çok sayıda ölçüm alınmış olması, o hastanın zaten ağır kabul edildiğinin göstergesi
olabilir. Bu bir klinik bulgu değil, bir bakım yoğunluğu vekilidir.

**Yanlış pozitif ikna edici mi?** Yanlış pozitif açıklamasını okuyan bir klinisyen
tahminin makul olduğuna ikna olur muydu? Olursa bu bir başarı değil, bir sorundur.

**Nedensellik sanılabilir mi?** Bu katkılardan hangileri nedensel bir iddia gibi
okunabilir ve bir klinisyen öyle davranırsa ne yanlış gider? Düşük sistolik basıncın
yüksek katkı vermesi, basıncı yükseltmenin yatış süresini kısaltacağı anlamına
gelmemektedir.

İstemin tam metni iki dilde `prompts/prompt-library.md` dosyasındadır.


## 5. İsteğe bağlı: SHAP · Optional

Bu adım zorunlu değildir ve atölye süresine göre atlanabilir. Doğrusal bir modelde SHAP,
yukarıda kesin olarak hesapladığımız katkıların aynısını üretmektedir. Değeri, doğrusal
olmayan bir modele geçtiğinizde aynı arayüzün çalışmaya devam etmesidir.

SHAP'ın Colab'da kurulumu yaklaşık bir dakika sürmektedir.


In [ ]:
# !pip -q install shap
# import shap
#
# X_transformed = model[:-1].transform(test[features])
# explainer = shap.LinearExplainer(model[-1], X_transformed)
# shap_values = explainer.shap_values(X_transformed)
# shap.summary_plot(shap_values, X_transformed,
#                   feature_names=model[:-1].get_feature_names_out())


---

## Bu adımın dersteki karşılığı

Derste açıklanabilirliğin ne verip ne vermediği anlatılmıştı. Verdiği: Hata avı, alt
grup denetimi ve klinik diyalog. Vermediği: Nedensellik, doğruluk güvencesi ve kararın
gerçek gerekçesi.

Rudin'in itirazı da burada anlamını bulmaktadır. Bu defterde kullandığımız model
doğrusal olduğu için açıklaması kesindir; kara kutu bir modele geçildiğinde aynı
grafikler üretilebilir ama artık bir yaklaşıklamadır ve özdeşlik denetimi tutmaz.
Yüksek riskli bir klinik kararda bu farkın bilinmesi gerekmektedir.

> Rudin C. Stop explaining black box machine learning models for high stakes decisions
> and use interpretable models instead. Nature Machine Intelligence. 2019;1:206-215.
> [doi:10.1038/s42256-019-0048-x](https://doi.org/10.1038/s42256-019-0048-x)



---

### Uyarı

Bu defterde üretilen hiçbir model doğrulanmış bir klinik araç değildir. MIMIC-IV demo
verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir yoğun bakım
popülasyonunu temsil etmemektedir. Buradaki çıktılar öğretim amaçlıdır.
